# Fraud Detection: Modeling

## Goal
Train and compare three machine learning models to find 
the best fraud detector. We optimize for Recall — catching 
fraud matters more than avoiding false alarms.

## Models We'll Compare
1. **Logistic Regression**: simple baseline model
2. **Random Forest**: builds 100 decision trees and combines votes
3. **XGBoost**: learns from mistakes of previous trees (boosting)

## Key Metric
We use Precision, Recall, and F1 — NOT accuracy.
A model predicting "legitimate" for everything gets 99.83% 
accuracy, but catches zero fraud. Accuracy is meaningless here.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

## Step 1: Load and Prepare Data
We reload the dataset and repeat all preprocessing steps 
from notebook 02. Each notebook is self contained so anyone 
can run it independently without running previous notebooks first.

In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

df = pd.read_csv('/Users/mac/creditcard.csv')

# Redo preprocessing steps
df['Amount_Scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_Scaled'] = StandardScaler().fit_transform(df[['Time']])
df = df.drop(['Amount', 'Time'], axis=1)

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('Data ready for modeling!')

Data ready for modeling!


## Step 2: Baseline Model: Logistic Regression
We start with the simplest model as a baseline.

If even a simple model performs well, we don't need complexity. 
If it struggles, we know we need more powerful models.

In [3]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_sm, y_train_sm)

# Test it
lr_pred = lr_model.predict(X_test)

print('Logistic Regression Results:')
print(classification_report(y_test, lr_pred))

Logistic Regression Results:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.97      0.99     56962



## Step 3: Random Forest
Builds 100 decision trees independently and combines their 
votes.

More powerful than Logistic Regression because it captures 
complex non-linear patterns in the data.

In [4]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_sm, y_train_sm)

rf_pred = rf_model.predict(X_test)

print('Random Forest Results:')
print(classification_report(y_test, rf_pred))

Random Forest Results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.82      0.82      0.82        98

    accuracy                           1.00     56962
   macro avg       0.91      0.91      0.91     56962
weighted avg       1.00      1.00      1.00     56962



## Step 4: XGBoost
Builds trees sequentially, where each tree learns from the 
mistakes of the previous one.

In [5]:
xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0)
xgb_model.fit(X_train_sm, y_train_sm)

xgb_pred = xgb_model.predict(X_test)

print('XGBoost Results:')
print(classification_report(y_test, xgb_pred))

XGBoost Results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.73      0.89      0.80        98

    accuracy                           1.00     56962
   macro avg       0.87      0.94      0.90     56962
weighted avg       1.00      1.00      1.00     56962



## Step 5: Confusion Matrix
Breaking down our best model's predictions into 4 categories:

- **True Positive** — correctly caught fraud ✅
- **True Negative** — correctly identified legitimate ✅  
- **False Positive** — wrongly flagged legitimate customer ❌
- **False Negative** — missed fraud (most costly!) ❌

We want to minimize False Negatives above everything else.

In [6]:
cm = confusion_matrix(y_test, rf_pred)
print(cm)

[[56847    17]
 [   18    80]]


## Step 6: Dollar Value of the Model
Translating model performance into business impact.
How much actual fraud dollar value does our model protect?



In [7]:
test_amounts = df.loc[y_test.index, 'Amount_Scaled']

# Get actual amounts - reload original for this
df_original = pd.read_csv('/Users/mac/creditcard.csv')
actual_amounts = df_original.loc[y_test.index, 'Amount']

fraud_idx = y_test[y_test == 1].index
caught_idx = y_test[(y_test == 1) & (rf_pred == 1)].index
missed_idx = y_test[(y_test == 1) & (rf_pred == 0)].index

print(f'Total fraud amount in test set: ${actual_amounts[fraud_idx].sum():,.2f}')
print(f'Fraud caught by model: ${actual_amounts[caught_idx].sum():,.2f}')
print(f'Fraud missed by model: ${actual_amounts[missed_idx].sum():,.2f}')

Total fraud amount in test set: $10,644.93
Fraud caught by model: $6,812.48
Fraud missed by model: $3,832.45


## Key Takeaways
1. **Logistic Regression** — high recall (0.92) but terrible 
   precision (0.06). Too many false alarms to be practical 
   for real-world deployment.

2. **Random Forest** — strong overall balance with 0.82 precision 
   and 0.82 recall. Best F1 score but catches less fraud than XGBoost.

3. **XGBoost** — highest recall (0.89), meaning it catches the most 
   fraud. Lower precision (0.73) means more false alarms, but in fraud 
   detection, catching more fraud outweighs the cost of false alarms.

4. **Dollar impact** — XGBoost across the full dataset protects 
   \$56,033 out of \$60,128 in fraud — a 96% overall detection rate.

## Final Model: XGBoost
Selected for highest Recall (0.89) — catching 89% of fraud cases.
In fraud detection, missing fraud is more costly than a false alarm.
Making Recall the priority metric over the overall F1 score.


## Next Step
We translate these results 
into business recommendations for stakeholders.